# 01 - EDA (Exploration des données)


In [ ]:
import sys, os
sys.path.append(os.path.abspath('../src'))
print(sys.path)

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from extract import read_source_csv, rename_columns_for_staging

pd.set_option('display.max_columns', None)

## 1. Chargement du fichier source brut

In [ ]:
df = read_source_csv('../data/raw/Sample - Superstore.csv')
df = rename_columns_for_staging(df)
print(df.shape)
df.head()

## 2. Structure générale : types, valeurs non nulles

In [ ]:
df.info()

## 3. Valeurs manquantes par colonne

In [ ]:

df.isna().sum().sort_values(ascending=False)

## 4. Doublons

On regarde 2 choses : les doublons EXACTS (toutes colonnes identiques), et les doublons sur row_id (qui est censé être unique).

In [ ]:
print('Doublons exacts (toutes colonnes) :', df.duplicated().sum())
print('Doublons sur row_id :', df.duplicated(subset=['row_id']).sum())

## 5. Statistiques descriptives des colonnes numériques

In [3]:
df_num = df[['sales', 'quantity', 'discount', 'profit']].apply(pd.to_numeric, errors='coerce')
df_num.describe()

,sales,quantity,discount,profit
count,9.863000e+03,9865.000000,10064.000000,10064.000000
mean,1.950322e+03,3.780233,0.157501,28.571309
std,4.410673e+04,2.240134,0.210586,233.509461
min,4.440000e-01,-5.000000,0.000000,-6599.978000
25%,1.729000e+01,2.000000,0.000000,1.724800
50%,5.450000e+01,3.000000,0.200000,8.643600
75%,2.107600e+02,5.000000,0.200000,29.331600
max,1.131924e+06,14.000000,1.500000,8399.976000


## 6. Valeurs aberrantes potentielles (règles de gestion du brief)

In [ ]:
print('Discount > 1 (>100%)   :', (df_num['discount'] > 1).sum())
print('Quantity < 0            :', (df_num['quantity'] < 0).sum())
print('Profit négatif (normal, à ne PAS traiter comme anomalie) :', (df_num['profit'] < 0).sum())

## 7. Tendances par catégorie / région / segment

In [ ]:
df_tmp = df.copy()
df_tmp['sales'] = df_num['sales']
df_tmp['category'] = df_tmp['category'].str.strip().str.title()

ventes_categorie = df_tmp.groupby('category')['sales'].sum().sort_values(ascending=False)
ventes_categorie

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(x=ventes_categorie.index, y=ventes_categorie.values, ax=ax)
ax.set_title('Ventes totales par catégorie')
ax.set_ylabel('Sales ($)')
plt.show()

In [ ]:
ventes_region = df_tmp.groupby('region')['sales'].sum().sort_values(ascending=False)
ventes_region

In [ ]:
# df_tmp['segment'] = df_tmp['segment'].str.strip().str.title()
ventes_segment = df_tmp.groupby('segment')['sales'].sum().sort_values(ascending=False)
ventes_segment

## 8. Corrélations entre variables numériques

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(df_num.corr(), annot=True, cmap='coolwarm')
plt.title('Matrice de corrélation')
plt.show()